# 第 2 天作业 —— 人工智能记者

## 练习目标（理念）

做一个「本地新闻摘要员」：抓取新闻站正文，交给本地 **Ollama**（OpenAI 兼容接口）写成 Markdown 摘要。

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `fetch_website_contents(...)`（来自 `scraper`） |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定「记者人设」，user 放网页正文 |
| 本地模型 via OpenAI SDK | `base_url=http://localhost:11434/v1`，`model='llama3.2'` |

## 怎么跑

1. 确保本机已启动 Ollama，并拉取 `llama3.2`
2. 同目录能导入 `scraper.fetch_website_contents`
3. 从上到下运行单元格；默认抓取 `https://www.bbc.com/news`


In [ ]:
# ========== 导入：抓取工具 + 展示 + OpenAI 兼容客户端 ==========

# 从本地 scraper 模块导入 fetch_website_contents：把网页正文拉成字符串
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown
from IPython.display import display, Markdown
# 从 openai 导入 OpenAI 客户端类：这里用来打本地 Ollama 的 OpenAI 兼容接口
from openai import OpenAI

# Ollama 的 OpenAI 兼容基址：/v1 表示走 chat.completions 那一套 API 形状
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# system prompt：定「客观专业记者」人设；发给模型的指令保持英文（改译会改变行为）
system_prompt = """
You are a objective , professional journalist. You get the content of the news page and you write a summarization about what is happening in the world lately. Ignoring text that might be navigation related. Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown. """
# user prompt 前缀：告诉模型「下面是网页内容，请做最新新闻摘要」
user_prompt = """
    Here are the contents of a website. Give me a summary of latest news
"""

# 抓取 BBC News 首页正文（导航类噪音靠 system prompt 要求忽略）
news_website = fetch_website_contents("https://www.bbc.com/news")

# ========== 组装 messages：system 定角色，user = 前缀 + 网页正文 ==========
messages = [{"role" : "system",
             "content": system_prompt},
            {"role": "user",
             "content": user_prompt + news_website}] # fill this in

# ========== 调用本地模型：OpenAI SDK 指向 Ollama ==========
# api_key='ollama'：兼容接口常需要占位密钥；真正鉴权由本地 Ollama 处理
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# model 名必须与本机 ollama list 里一致；messages 即上面拼好的对话
response = openai.chat.completions.create(model='llama3.2',
                                          messages= messages)


# ========== 取出回复并在笔记本里用 Markdown 展示 ==========
# choices[0].message.content：非流式时整段文本在这里
news_summary = response.choices[0].message.content
display(Markdown(news_summary))
